# Inspecting a board: score, decision, and where the defect is

`01_train.ipynb` produced a model that scores how far a board sits from what a correct one looks
like. A score is not yet an inspection station. This notebook turns it into two things an operator
can act on:

- **a decision**, by comparing the score against a limit fixed for that product
- **a location**, by taking the highest peaks of the anomaly map

You do not need to have run the training notebook. The cell below downloads the published
checkpoint, which was fitted on all four VisA circuit-board products, 3,614 good boards, no defects
and no masks.

**What you need.** The checkpoint download needs the GitHub release to exist, and the cell below
says so plainly if it does not. A CUDA GPU is optional here, unlike in the training notebook: the
runtime falls back to CPU and tells you which one it picked, but a board takes about 50 ms on an
RTX 5090 and considerably longer without one.

In [ ]:
import sys, subprocess
from pathlib import Path

HERE = Path.cwd()
REPO = next((p for p in [HERE, *HERE.parents] if (p / "pcb_anomaly").is_dir()), None)
if REPO is None:
    raise SystemExit(f"no repository root above {HERE}. Start Jupyter inside the clone.")
sys.path.insert(0, str(REPO))

CATEGORY = "pcb3"

# The checkpoint. Captured rather than inherited, because Jupyter does not show fd 1 from a child
# process, and fetch_checkpoint writes the useful part of a failure to stderr.
if not (REPO / "artifacts/detector.pt").exists():
    r = subprocess.run([sys.executable, str(REPO / "scripts/fetch_checkpoint.py")],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr, sep="")
    if r.returncode:
        raise SystemExit("could not fetch the checkpoint, see the message above")

# The data. This notebook is meant to stand alone, so it downloads what it needs rather than
# assuming 01 was run first.
if not sorted((REPO / "data/visa" / CATEGORY / "test/bad").glob("*.png")):
    print(f"downloading VisA {CATEGORY}, several GB on first run, and nothing prints until it "
          f"finishes ...", flush=True)
    r = subprocess.run([sys.executable, str(REPO / "scripts/get_visa.py"),
                        "--categories", CATEGORY], capture_output=True, text=True)
    print(r.stdout[-2000:] or "(no output)")
    if r.returncode:
        raise SystemExit(f"get_visa.py failed:\n{r.stderr[-2000:]}")

import cv2, numpy as np, matplotlib.pyplot as plt
from pcb_anomaly import load, inspect
from pcb_anomaly.render import boxes, heatmap

STATE = load()

## The accept/reject limit

The limit is set **per product**, at the 95th percentile of that product's good-board scores,
computed on a calibration half of the good images. **No defect score is ever consulted when choosing
it**, which is what makes the procedure runnable before you have collected a single defect. That is
the practical difference between this approach and a supervised one: you can commission the station
on day one.

In [ ]:
for part, limit in STATE["thr"]["per_board_p95"].items():
    print(f"  {part}   limit {limit:.4f}")

## One defective board

The heatmap is the model's raw output: how far each patch sits from what the model expects to see
there. The boxes are what an operator would actually be shown, drawn from the peaks of that map
after it has been scaled back up to the size of the photograph.

Peaks are restricted to the board itself. A hot pixel on the background is not a defect in the part,
and letting one win would put a box on the bench rather than on the unit.

In [ ]:
bad = sorted((REPO / "data/visa" / CATEGORY / "test/bad").glob("*.png"))
img = cv2.imread(str(bad[0]))

r = inspect(img, board=CATEGORY)
print(f"  score     {r['score']:.4f}")
print(f"  limit     {r['threshold']:.4f}")
print(f"  verdict   {r['verdict']}")
print(f"  peaks     {r['peaks']}")
print(f"  time      {r['ms_total']} ms   (first call, includes warm-up; steady state is ~50 ms on a GPU)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB));                 axes[0].set_title("the unit under test")
axes[1].imshow(cv2.cvtColor(heatmap(img, r["_raw_map"]), cv2.COLOR_BGR2RGB));   axes[1].set_title("anomaly map")
# boxes() is only meaningful on a reject: candidate boxes are the station saying "look here",
# and on a board that passed there is nothing to look at.
marked = boxes(img, r["peaks"], r["_half"]) if r["verdict"] == "FAIL" else img
axes[2].imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB));              axes[2].set_title(f"regions marked  ({r['verdict']})")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## The board with nothing wrong with it

This is the case that decides whether a station is usable. Finding a defect is the easy half. Not
inventing one is the half that determines whether an operator still trusts the marks in week two.

Note what the heatmap looks like here. Heatmaps are normalised per panel, so on a board with no
defect there is no peak to find and normalisation stretches ordinary noise across the full colour
range. It looks alarming and the score says otherwise. **Read the score, not the colour.**

In [ ]:
good = sorted((REPO / "data/visa" / CATEGORY / "test/good").glob("*.png"))
img_g = cv2.imread(str(good[0]))
rg = inspect(img_g, board=CATEGORY)

print(f"  score     {rg['score']:.4f}   (limit {rg['threshold']:.4f})")
print(f"  verdict   {rg['verdict']}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(img_g, cv2.COLOR_BGR2RGB));        axes[0].set_title("a board that passed")
axes[1].imshow(cv2.cvtColor(heatmap(img_g, rg["_raw_map"]), cv2.COLOR_BGR2RGB));  axes[1].set_title("its anomaly map, normalised")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## Across the whole split

One board proves nothing. Run every held-out image for this product and count. Read the false-alarm
figure as the friendlier of the two numbers: half of these good boards are the calibration half that
set the limit in the first place, so only the other half is genuinely held out from it. The
defective boards are held out from everything.

In [ ]:
rows = []
for f in bad:
    rows.append(("defective", inspect(cv2.imread(str(f)), board=CATEGORY)))
for f in good:
    rows.append(("good", inspect(cv2.imread(str(f)), board=CATEGORY)))

n_bad  = sum(1 for t, _ in rows if t == "defective")
n_good = sum(1 for t, _ in rows if t == "good")
caught = sum(1 for t, r in rows if t == "defective" and r["verdict"] == "FAIL")
false_alarm = sum(1 for t, r in rows if t == "good" and r["verdict"] == "FAIL")

print(f"  defective boards rejected   {caught:3d} of {n_bad:3d}   ({caught / n_bad:.1%})")
print(f"  good boards falsely rejected {false_alarm:3d} of {n_good:3d}   ({false_alarm / n_good:.1%})")

### The limit is a dial, not a constant

Which end of it you want depends on whether a missed defect reaching the customer or an
operator-minute costs you more, and that choice belongs to whoever owns the cost of a miss. Sweeping
it shows the trade directly.

In [ ]:
scores_bad  = np.array([r["score"] for t, r in rows if t == "defective"])
scores_good = np.array([r["score"] for t, r in rows if t == "good"])

print(f"  {'limit at':>16}  {'value':>8}  {'caught':>8}  {'false alarms':>13}")
for q in [90, 95, 99, 100]:
    thr = np.percentile(scores_good, q)
    print(f"  {'p' + str(q) + ' of good':>16}  {thr:8.4f}  "
          f"{(scores_bad >= thr).mean():7.1%}  {(scores_good >= thr).mean():12.1%}")

## What this does not cover

Two honest limits, because results on public data are easy to over-read.

**The model learns what a good unit looks like including how it is lit and framed.** Any variation
you allow in lighting or camera position becomes variation it must treat as normal, and that costs
sensitivity to the small differences that matter. A fixed station with controlled lighting is part
of the method, not an optional extra.

**Some of the separation on this benchmark is not the defect.** On VisA circuit boards, background
brightness alone separates good from defective at 0.68 to 0.77 AUROC. A properly controlled capture
station removes that shortcut, and some of the headline numbers with it.